In [1]:
# Parameters
import os, json
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
dry_run = False


In [2]:
# -- LAUNCH•EVENING: tickers that jump at the 15:59 -> 16:00 transition and snap straight back ----
#
# EVENING twin of LAUNCH•MORNING: the ONLY difference is the clock - the jump is the 16:00 bar (REF window 15:35-15:38,
# pre-check bar 15:59, reversion checked at the 16:01 / 16:02 closes). NOTE the 16:00 bar carries the closing auction,
# and 16:01/16:02 are after-hours bars that print rarely, so fewer events are judgeable than in the morning.
#
# ONE rating and ONE alpha per ticker - long and short are NOT split, up-moves and down-moves are pooled.
#
# EVENT (per ticker-day, per direction):
#   REF     = close of the LAST bar inside 15:35-15:38. No print there -> no event that day (no stale fallback).
#   JUMP    = the 16:00 bar. up = (h / REF - 1) * 100,  down = (l / REF - 1) * 100.
#             An event is a move of at least THRESH_PCT (0.7%) UP or DOWN. A day can hold both.
#   BAR     = the 16:00 bar must have traded at least MIN_SPIKE_V (10,000) shares - the only liquidity gate.
#   AT 16:00 (not before): if the 15:59 bar had ALREADY made the same +/-0.7% move against REF, the jump
#             began before the transition and that side of that day is not an event.
#   NO PRICE FLOOR and no dollar-volume floor - those gates were removed on purpose.
#
# REVERTED  = the move came back almost immediately AND almost all the way: at the 16:01 close OR the 16:02 close
#             at least RETRACE_OK (0.9) of the excursion is given back - the price is back to within 10% of the
#             move's size from REF, i.e. almost at the print that stood BEFORE the move (REF = the last close in
#             15:35-15:38),
#               up:   (h - c) / (h - REF)       down: (c - l) / (REF - l)
#             1.0 = back at REF, >1 = through REF, <0 = kept going. "16:01 or 16:02" = the better of the two.
#
# total    = events that can be judged: at least one of the 16:01 / 16:02 bars printed. The tape is sparse, a
#            missing bar means "no print", NOT "price unchanged", so such events are not counted at all.
# rating   = reverted / total  (0..1), up and down events together.
# alpha    = sqrt(mean(m^2)) over ALL events of the ticker, up and down together, m = size of the move in
#            percent (|up| or |down|, always positive) - one root-mean-square number for both directions.
# sigma    = the per-ticker `sigma` column of final.parquet (latest non-null value) - read, not computed.
import math
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

THRESH_PCT = 0.7
MIN_SPIKE_V = 10000
RETRACE_OK = 0.9           # 1.0 = exactly back at REF; 0.9 = within 10% of the move's size of REF
REF_FROM, REF_TO = "15:35", "15:38"
PRE_BAR, JUMP_BAR = "15:59", "16:00"
CHECK_BARS = ("16:01", "16:02")
WINDOW_SESSIONS = 65      # ratings, alpha and total are computed on the LAST 65 trading sessions only


def load_window_bars(final_path: str) -> pd.DataFrame:
    """Only the 15:35-16:02 bars, straight from final.parquet row groups (a few hundred thousand rows)."""
    pf = pq.ParquetFile(final_path)
    parts = []
    for gi in range(pf.num_row_groups):
        t = pf.read_row_group(gi, columns=["ticker", "dt", "h", "l", "c", "v"])
        hm = pc.utf8_slice_codeunits(t["dt"], 11, 16)
        t = t.filter(pc.and_(pc.greater_equal(hm, REF_FROM), pc.less_equal(hm, CHECK_BARS[-1])))
        if t.num_rows:
            parts.append(t.to_pandas())
    if not parts:
        return pd.DataFrame(columns=["ticker", "dt", "h", "l", "c", "v", "date", "hm"])
    d = pd.concat(parts, ignore_index=True)
    d["ticker"] = d["ticker"].astype(str).str.strip().str.upper()
    d = d.drop_duplicates(["ticker", "dt"])          # final.parquet can carry exact duplicate rows
    d["date"] = d["dt"].str.slice(0, 10)
    d["hm"] = d["dt"].str.slice(11, 16)
    return d


def load_sigma(final_path: str) -> pd.Series:
    """Per-ticker `sigma` from final.parquet: the LATEST non-null value (row groups are in file order)."""
    pf = pq.ParquetFile(final_path)
    if "sigma" not in pf.schema_arrow.names:
        raise KeyError("final.parquet has no 'sigma' column")
    out = {}
    for gi in range(pf.num_row_groups):
        t = pf.read_row_group(gi, columns=["ticker", "sigma"]).to_pandas()
        t = t.dropna(subset=["sigma"])
        if t.empty:
            continue
        t["ticker"] = t["ticker"].astype(str).str.strip().str.upper()
        for tk, sv in t.drop_duplicates("ticker", keep="last")[["ticker", "sigma"]].itertuples(index=False):
            out[tk] = float(sv)
    return pd.Series(out, name="sigma").rename_axis("ticker")


def build_events(d: pd.DataFrame) -> pd.DataFrame:
    """One row per EVENT (ticker-day-direction): size in percent and the best retrace over 16:01/16:02."""
    ref = (d[(d.hm >= REF_FROM) & (d.hm <= REF_TO)].sort_values("dt")
             .groupby(["ticker", "date"])["c"].last().rename("ref"))
    j = (d[d.hm == JUMP_BAR].groupby(["ticker", "date"])
           .agg(hi=("h", "max"), lo=("l", "min"), v0=("v", "sum")))
    p = d[d.hm == PRE_BAR].groupby(["ticker", "date"]).agg(hi59=("h", "max"), lo59=("l", "min"))
    ck = {b: d[d.hm == b].set_index(["ticker", "date"])["c"].rename(f"c{k}") for k, b in enumerate(CHECK_BARS, 1)}
    ev = j.join(ref, how="inner").join(p, how="left").join(ck[CHECK_BARS[0]], how="left").join(ck[CHECK_BARS[1]], how="left")
    ev = ev[(ev["ref"] > 0) & (ev["v0"] >= MIN_SPIKE_V)].reset_index()

    rows = []
    for side in ("up", "dn"):
        if side == "up":
            m = (ev["hi"] / ev["ref"] - 1.0) * 100.0
            pre = (ev["hi59"] / ev["ref"] - 1.0) * 100.0
            ok = (m >= THRESH_PCT) & ~(pre >= THRESH_PCT).fillna(False)
            ext, den = ev["hi"], ev["hi"] - ev["ref"]
            rets = [(ext - ev[c]) / den for c in ("c1", "c2")]
        else:
            m = (1.0 - ev["lo"] / ev["ref"]) * 100.0
            pre = (1.0 - ev["lo59"] / ev["ref"]) * 100.0
            ok = (m >= THRESH_PCT) & ~(pre >= THRESH_PCT).fillna(False)
            ext, den = ev["lo"], ev["ref"] - ev["lo"]
            rets = [(ev[c] - ext) / den for c in ("c1", "c2")]
        with np.errstate(divide="ignore", invalid="ignore"):
            best = pd.concat(rets, axis=1).max(axis=1, skipna=True)      # NaN when neither bar printed
        rows.append(pd.DataFrame({"ticker": ev["ticker"], "date": ev["date"], "side": side,
                                  "move_pct": m, "retrace": best, "v0": ev["v0"]})[ok])
    return pd.concat(rows, ignore_index=True)


def build_table(events: pd.DataFrame, sigma: pd.Series) -> pd.DataFrame:
    """ticker, rating, total, sigma, alpha - nothing else."""
    rows = []
    for tk, g in events.groupby("ticker"):
        r = g["retrace"].dropna()
        if r.empty:
            continue                                    # nothing judgeable -> no rating, so no row
        m = g["move_pct"].to_numpy(float)
        rows.append({"ticker": tk, "rating": float((r >= RETRACE_OK).mean()), "total": int(len(r)),
                     "sigma": float(sigma[tk]) if tk in sigma.index else np.nan,
                     "alpha": float(np.sqrt(np.mean(m * m)))})
    return pd.DataFrame(rows, columns=["ticker", "rating", "total", "sigma", "alpha"]).sort_values("ticker").reset_index(drop=True)


In [3]:
# -- self-test on hand-built bars (runs WITHOUT final.parquet) ---------------------------------------------
def _bars(rows):
    rows = [r if len(r) == 6 else (*r, 50_000.0) for r in rows]        # default v = 50k shares per bar
    d = pd.DataFrame(rows, columns=["ticker", "dt", "h", "l", "c", "v"])
    d["date"] = d["dt"].str.slice(0, 10)
    d["hm"] = d["dt"].str.slice(11, 16)
    return d


def _selftest():
    D1, D2, D3, D4 = "2026-09-01", "2026-09-02", "2026-09-03", "2026-09-04"
    rows = []
    # AAA d1: ref 10.00 (15:37 is the LAST bar in the window, not 15:35), UP to 10.50 (+5%); 16:01 close 10.45
    #   (retrace 0.1 - not enough) but 16:02 close 10.03 (retrace 0.94) -> reverted via 16:02
    rows += [("AAA", f"{D1} 15:35:00", 9, 9, 9.5), ("AAA", f"{D1} 15:37:00", 10, 10, 10.0),
             ("AAA", f"{D1} 15:59:00", 10.02, 10.0, 10.01), ("AAA", f"{D1} 16:00:00", 10.5, 10.0, 10.4),
             ("AAA", f"{D1} 16:01:00", 10.5, 10.3, 10.45), ("AAA", f"{D1} 16:02:00", 10.4, 10.0, 10.03)]
    # AAA d2: DOWN to 9.40 (-6%), 16:01 close 9.45 (retrace 0.08), 16:02 close 9.42 -> NOT reverted
    # AAA d5 below: a 50% retrace - reverted under the old 0.5 rule, NOT any more
    rows += [("AAA", f"{D2} 15:36:00", 10, 10, 10.0), ("AAA", f"{D2} 16:00:00", 10.0, 9.4, 9.43),
             ("AAA", f"{D2} 16:01:00", 9.5, 9.4, 9.45), ("AAA", f"{D2} 16:02:00", 9.45, 9.4, 9.42)]
    # AAA d3: only +0.5% -> below 0.7%, not an event
    rows += [("AAA", f"{D3} 15:36:00", 10, 10, 10.0), ("AAA", f"{D3} 16:00:00", 10.05, 10.0, 10.02),
             ("AAA", f"{D3} 16:01:00", 10.05, 10.0, 10.02)]
    # AAA d4: +2% jump but NO 16:01 / 16:02 print -> not judgeable: not in total, but still in alpha
    rows += [("AAA", f"{D4} 15:36:00", 10, 10, 10.0), ("AAA", f"{D4} 16:00:00", 10.2, 10.0, 10.1)]
    rows += [("AAA", "2026-09-05 15:36:00", 10, 10, 10.0), ("AAA", "2026-09-05 16:00:00", 10.4, 10.0, 10.3),
             ("AAA", "2026-09-05 16:01:00", 10.3, 10.0, 10.2), ("AAA", "2026-09-05 16:02:00", 10.3, 10.0, 10.2)]
    # BBB: +10% on only 5,000 shares -> below the 10,000-share bar gate -> nothing
    rows += [("BBB", f"{D1} 15:36:00", 10, 10, 10.0), ("BBB", f"{D1} 16:00:00", 11.0, 10.0, 10.9, 5000.0),
             ("BBB", f"{D1} 16:01:00", 10.2, 10.0, 10.1)]
    # CCC: no REF print -> nothing
    rows += [("CCC", f"{D1} 15:20:00", 5, 5, 5.0), ("CCC", f"{D1} 16:00:00", 9, 5, 8), ("CCC", f"{D1} 16:01:00", 6, 5, 5.5)]
    # DDD: 15:59 bar ALREADY +5% over REF -> jump began before 16:00 -> no UP event
    rows += [("DDD", f"{D1} 15:36:00", 10, 10, 10.0), ("DDD", f"{D1} 15:59:00", 10.5, 10.0, 10.4),
             ("DDD", f"{D1} 16:00:00", 10.8, 10.3, 10.5), ("DDD", f"{D1} 16:01:00", 10.5, 10.0, 10.1)]
    # EEE: a cheap $1 stock - must NOT be filtered by price any more
    rows += [("EEE", f"{D1} 15:36:00", 1.0, 1.0, 1.0), ("EEE", f"{D1} 16:00:00", 1.05, 1.0, 1.04),
             ("EEE", f"{D1} 16:01:00", 1.03, 1.0, 1.004)]
    ev = build_events(_bars(rows))
    assert set(ev["ticker"]) == {"AAA", "EEE"}, ev
    sigma = pd.Series({"AAA": 1.5, "EEE": 0.4}, name="sigma")
    t = build_table(ev, sigma).set_index("ticker")
    a, e = t.loc["AAA"], t.loc["EEE"]
    assert a.total == 3 and abs(a.rating - 1 / 3) < 1e-12, a                    # d1 reverted (16:02); d2 and d5(50%) not; d4 unjudged
    assert abs(a.alpha - math.sqrt((5.0 ** 2 + 6.0 ** 2 + 2.0 ** 2 + 4.0 ** 2) / 4)) < 1e-9, a.alpha   # RMS over up+down AND the unjudged event
    assert a.sigma == 1.5 and e.total == 1 and e.rating == 1.0 and abs(e.alpha - 5.0) < 1e-9, e
    assert list(t.columns) == ["rating", "total", "sigma", "alpha"]
    print("self-test OK: ref = last bar in 15:35-15:38, 0.7% up OR down, 10k-share bar, 16:01-or-16:02 reversion to within 10% of REF, "
          "one pooled rating + alpha, no price floor")


_selftest()


self-test OK: ref = last bar in 15:35-15:38, 0.7% up OR down, 10k-share bar, 16:01-or-16:02 reversion to within 10% of REF, one pooled rating + alpha, no price floor


In [4]:
# -- run on final.parquet ---------------------------------------------------------------------------------
# PATHS: identical to OPG-reversal's runner - FINAL_PARQUET_PATH / SIGNALS_DIR win, else ORION_HOME, else the
# OriON folder found by walking up from the cwd. Output lands in <OriON>/signals/launchevening/ - the tree the runner
# publishes - NOT in "../signals" relative to notebooks/ (that one is STRATEGIES\signals, which is gitignored
# and never published).
from pathlib import Path
import gzip, datetime


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break
        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")
    return final_path, out_dir


FINAL_PATH, OUT_DIR = _resolve_orion_paths("launchevening")

bars = load_window_bars(str(FINAL_PATH))
_sessions = sorted(bars["date"].unique())[-WINDOW_SESSIONS:]
bars = bars[bars["date"].isin(_sessions)]                      # the last 65 trading sessions, nothing older
events = build_events(bars)
sigma = load_sigma(str(FINAL_PATH))
table = build_table(events, sigma)

print(f"sessions: {len(_sessions)} ({_sessions[0]} .. {_sessions[-1]}) | events (>= {THRESH_PCT}% up or down, bar >= {MIN_SPIKE_V:,} shares): "
      f"{len(events):,} ({int((events.side == 'up').sum()):,} up / {int((events.side == 'dn').sum()):,} down)")
print(f"tickers in the file: {len(table):,} | without sigma in final.parquet: {int(table['sigma'].isna().sum()):,}")

if not dry_run:
    table.round({"rating": 4, "sigma": 4, "alpha": 4}).to_csv(OUT_DIR / "launchevening_tickers.csv", index=False)
    print("written ->", OUT_DIR / "launchevening_tickers.csv")
print(table.sort_values(["total", "rating"], ascending=False).head(15).round(3).to_string(index=False))


sessions: 65 (2026-07-28 .. 2026-10-27) | events (>= 0.7% up or down, bar >= 10,000 shares): 4,569 (2,608 up / 1,961 down)
tickers in the file: 1,033 | without sigma in final.parquet: 6
written -> C:\datum-api-examples-main\OriON\signals\launchevening\launchevening_tickers.csv
ticker  rating  total  sigma  alpha
  QBTS   0.083     24  2.059  1.131
  QUBT   0.250     20  2.083  1.076
  AAOI   0.053     19  3.685  1.148
  CRML   0.278     18  1.766  1.244
  APLD   0.056     18  3.023  1.029
  WULF   0.059     17  3.082  1.024
  CLSK   0.250     16  2.733  1.109
  NVTS   0.333     15  3.020  1.060
  VIXY   0.545     11  1.401  3.337
  CBRS   0.200     10  2.279  0.873
  RGTI   0.200     10  2.071  0.920
  AXTI   0.100     10  3.528  1.297
  SOAR   0.778      9  3.061  2.673
  XNDU   0.444      9  2.519  1.167
  USAR   0.500      8  1.764  1.080


In [5]:
# -- PUBLISH: signals/launchevening/launchevening_rolling_perf.json.gz ---------------------------------------------
# The EVENING twin of LAUNCH-MORNING's rolling-perf file, in the SAME schema (meta + rows[ticker, alpha_pos, alpha_neg,
# sigma, lambda, ratings[], trades[]], cls UPPERCASE, sign "pos" / "neg"). NO bridge strategy reads it yet - when one is
# built, its parser can be a straight port of MorLaunchRollingPerfParser with the class list {"EVE1600"}.
#
# HOW THIS NOTEBOOK'S NUMBERS ARE MAPPED (read before trusting a field by its name):
#   * ONE rating per ticker, so the SAME cell is written under both "pos" and "neg" - the gate looks up the sign of
#     the live move, and for either sign it finds the same pooled numbers. alpha_pos == alpha_neg == alpha.
#   * total / wins / win_rate / win_rate_lb: total = judgeable events, wins = events that came back >= 90% by the
#     16:01 / 16:02 close, win_rate = wins / total, win_rate_lb = its Wilson lower bound. THIS IS A REVERSION RATE,
#     NOT A PNL WIN RATE - the bridge's MinRate / MinTotal then gate tickers on how reliably they snap back.
#   * gamma = THRESH_PCT (0.7), gamma_n = total, gamma_rate = win_rate. The gate REJECTS a ticker whose cell has no
#     gamma ("entry.Gamma is not {} gamma"), and it compares the live |move| against it, so the honest value is the
#     event threshold this notebook itself used. It is NOT fitted per ticker.
#   * mean_pnl / pnl_lb / lambda are NOT produced here -> null. trades is empty (no Scout page needs it).
#   * sigma = final.parquet's own `sigma`; the gate prefers a published sigma over the row's live one.
CLASS = "EVE1600"        # the one class of this strategy: the 16:00 jump
ROLLING_PATH = OUT_DIR / "launchevening_rolling_perf.json.gz"


def _num(v, nd=4):
    return None if v is None or (isinstance(v, float) and not math.isfinite(v)) else round(float(v), nd)


def _wilson_lb(k, n, z=1.96):
    if n <= 0:
        return None
    p = k / n
    den = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - r) / den


def build_rolling_perf(table: pd.DataFrame, sessions: list) -> dict:
    rows = []
    for r in table.itertuples(index=False):
        wins = int(round(r.rating * r.total))
        cell = {"cls": CLASS, "total": int(r.total), "wins": wins, "win_rate": _num(r.rating),
                "win_rate_lb": _num(_wilson_lb(wins, int(r.total))), "mean_pnl": None, "pnl_lb": None,
                "gamma": THRESH_PCT, "gamma_n": int(r.total), "gamma_rate": _num(r.rating)}
        rows.append({"ticker": r.ticker, "alpha_pos": _num(r.alpha), "alpha_neg": _num(r.alpha),
                     "sigma": _num(r.sigma), "lambda": None,
                     "ratings": [{**cell, "sign": "pos"}, {**cell, "sign": "neg"}], "trades": []})
    recent = [d.replace("-", "") for d in sessions]
    return {
        "meta": {
            "version": "launchevening-rolling-perf-v1",
            "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
            "most_recent_session": recent[-1] if recent else None,
            "sessions_seen": len(recent), "windows_days": [WINDOW_SESSIONS], "recent_dates": recent,
            "position_usd": 1000, "min_dev": THRESH_PCT,
            "pnl_basis": ("NOT a pnl. rating = share of events (16:00 bar >= %d shares moved >= %.1f%% up or down vs the last "
                          "close in 15:35-15:38) whose move was given back >= %d%% by the 16:01 or 16:02 close; up and down "
                          "pooled, the same cell under pos and neg; gamma = the %.1f%% event threshold; mean_pnl, pnl_lb and "
                          "lambda unpublished." % (MIN_SPIKE_V, THRESH_PCT, int(RETRACE_OK * 100), THRESH_PCT)),
            "checkpoints": {"JUMP_BAR": JUMP_BAR, "CHECK_BARS": list(CHECK_BARS), "REF_WINDOW": [REF_FROM, REF_TO]},
        },
        "rows": rows,
    }


def validate_published(path) -> None:
    """Re-reads the file the way the C# parser will: strict JSON (no NaN), depth-correct, every row has both signs."""
    with gzip.open(path, "rt", encoding="utf-8") as f:
        doc = json.loads(f.read(), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f"non-finite {c} in file")))
    assert set(doc) == {"meta", "rows"} and doc["rows"], "meta/rows missing or empty"
    for r in doc["rows"]:
        assert r["ticker"] and {c["sign"] for c in r["ratings"]} == {"pos", "neg"}, r["ticker"]
        for c in r["ratings"]:
            assert c["cls"] == CLASS and c["gamma"] is not None and c["total"] >= 1 and 0 <= c["win_rate"] <= 1, r["ticker"]
    print(f"validated: {len(doc['rows']):,} tickers, both signs, gamma present, strict JSON")


# selftest of the mapping on a two-ticker table (no files needed)
_demo = build_rolling_perf(pd.DataFrame({"ticker": ["AAA", "BBB"], "rating": [0.75, 0.5], "total": [8, 2],
                                         "sigma": [1.5, np.nan], "alpha": [3.0, 1.2]}), ["2026-09-01", "2026-09-02"])
assert _demo["rows"][1]["sigma"] is None and _demo["rows"][0]["ratings"][0]["wins"] == 6
assert _demo["rows"][0]["alpha_pos"] == _demo["rows"][0]["alpha_neg"] == 3.0
assert _demo["meta"]["recent_dates"] == ["20260901", "20260902"] and _demo["meta"]["windows_days"] == [65]

if not dry_run:
    doc = build_rolling_perf(table, _sessions)
    with gzip.open(ROLLING_PATH, "wt", encoding="utf-8", newline="\n") as f:
        json.dump(doc, f, ensure_ascii=False, allow_nan=False)
    print("written ->", ROLLING_PATH)
    validate_published(ROLLING_PATH)


written -> C:\datum-api-examples-main\OriON\signals\launchevening\launchevening_rolling_perf.json.gz
validated: 1,033 tickers, both signs, gamma present, strict JSON
